In [1]:
# Parameters
nb_name = "ICT-35c-HumorDepthProfile-SAE-Python"
layers = {"early": 4, "mid": 12, "late": 19}
n_layers_model = 24
n_draws = 2048
seed = 42

> **Statut épistémique** — **Sans verdict à ce jour** : aucune ligne de la [matrice de dissociations](../../../docs/ict/dissociations-matrix.md) ne concerne ce notebook ; son statut épistémique sera porté par la matrice le cas échéant.

## ICT-35c -- HumorDepthProfile-SAE : le verdict survit-il a la profondeur ? (#14035, tranche 2)

La tranche 1 (**ICT-35b**, meme dossier) a mesure le differentiel humour->unfun sur 30 paires minimales au residu **12/24** : verdict **INCONCLUSIVE** -- amplitude du trained sous le bruit permute, top-5 features disjoints du controle. Une objection immediate : *un resultat a une seule profondeur ne dit rien de la profondeur*. La representation d'une incongruite comique peut etre precoce (lexicale, couches 3-8) ou tardive (pragmatique, couches 18-22) -- mesurer au milieu seulement peut rater l'une et l'autre.

Cette tranche reprend **le meme corpus, le meme protocole pre-registre, la meme graine** et fait varier **une seule chose** : la profondeur de capture. Trois couches -- early 4/24 (frac 0.17), mid 12/24 (tranche 1, frac 0.52), late 19/24 (frac 0.83) -- avec a chaque fois le couple trained + controle de permutation.

In [2]:
# -*- coding: utf-8 -*-
# Corpus + paires : chargement deterministe depuis l'organe humour_banc
# (extraction #20316 ; avant : reproduction par execution des cellules du banc 18c).
import sys
from pathlib import Path
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from ict.humor_pairs import (
    build_pairs, load_corpus_dur, validate_corpus, validate_pairs,
    measure_humor_differential,
)
corpus = load_corpus_dur()
validate_corpus(corpus)
pairs = build_pairs(corpus)
validate_pairs(pairs)
print("corpus:", len(corpus), "instances | paires:", len(pairs))

corpus: 120 instances | paires: 30


## Lecture 1 -- Trente paires, trois couches, deux variantes chacune

Le corpus et les paires sont ceux de la tranche 1 (exclusions documentees la-bas : `joke-p01`, `joke-p17`, instances Argumentum). Les captures early/late sont **neuves** (GPU local, `scripts/extract_sae_traces.py`, `--layer-frac 0.1667` et `0.8333`, seed 42, SAE-Res-W32K-L0_50) ; la capture mid est celle recommittee par la tranche 1. Le controle de chaque couche = permutation seedee des lignes d'input embeddings, meme graine.

Trois precisions de lecture sur ce chargement :

- **La paire est un triplet, pas un duo.** Chaque paire minimale engendre trois prompts : la version humour, la version unfun (le meme texte prive de sa chute), et une version **ctrl_edit** ou un changement lexical banal est applique a la meme zone. Le banc compte donc 30 paires x 3 prompts = 90 prompts -- c'est le compte affiche par le manifeste ci-dessus. Le troisieme prompt ne sert pas a mesurer le differentiel : il fournit l'echelle de comparaison `delta_ctrl` du verdict pre-registre (le differentiel humour doit depasser **1,5 fois** celui d'un banal changement lexical pour compter).
- **L0 = 50,00 sur les six traces n'est pas une coincidence.** L'extracteur demande les **top-k = 50** features les plus actives par token (k=50 dans le manifeste) : le SAE W32K-L0_50 est lu en mode epars fixe, exactement 50 atomes actifs par token, quel que soit le token ou la couche. C'est ce qui rend les vecteurs de zone comparables entre couches : meme discipline de lecture a chaque profondeur.
- **Les traces sont des artefacts committes, pas une capture au vol.** Les six fichiers `.npz` (3 couches x 2 jambes) portent leur manifeste (couche, fraction, seed 42, nombre de tokens) ; la cellule verifie ces champs par assertion avant toute mesure. La capture elle-meme est un passage GPU local documente (`scripts/extract_sae_traces.py`), rejouable au meme seed.

In [3]:
# Chargement des 6 traces : manifeste + L0 mesure par couche (meme lecture que ICT-35b).
import json
import numpy as np

metas, l0s = {}, {}
for tag, layer in layers.items():
    for variant in ("trained", "control"):
        p = ROOT / "traces" / f"humor35b_qwen35-2b-base_layer{layer}of{n_layers_model}_{variant}.npz"
        d = np.load(p, allow_pickle=False)
        meta = json.loads(str(d["__meta__"]))
        n_entries = sum(1 for k in d.files if k.endswith("__topk_ids"))
        l0 = np.concatenate([
            (d[k] > 0).sum(axis=1) for k in d.files if k.endswith("__topk_vals")
        ]).mean()
        assert meta["layer"] == layer and meta["n_layers"] == n_layers_model
        assert meta["seed"] == seed, f"seed manifestee {meta['seed']} != {seed}"
        metas[(tag, variant)] = meta
        l0s[(tag, variant)] = float(l0)
        print(f"{tag:5s} L{layer:2d}/24 {variant:7s}: {n_entries} prompts, "
              f"frac {meta['layer_frac']:.4f}, {meta['n_tokens_total']} tokens, L0={l0:.2f} (k={meta['k']})")

early L 4/24 trained: 90 prompts, frac 0.1739, 1868 tokens, L0=50.00 (k=50)
early L 4/24 control: 90 prompts, frac 0.1739, 1868 tokens, L0=50.00 (k=50)
mid   L12/24 trained: 90 prompts, frac 0.5217, 1868 tokens, L0=50.00 (k=50)
mid   L12/24 control: 90 prompts, frac 0.5217, 1868 tokens, L0=50.00 (k=50)
late  L19/24 trained: 90 prompts, frac 0.8261, 1868 tokens, L0=50.00 (k=50)
late  L19/24 control: 90 prompts, frac 0.8261, 1868 tokens, L0=50.00 (k=50)


## Lecture 2 -- Mesure pre-registree par couche : la jambe trained

La mesure est exactement celle pre-registree en tranche 1 (null croise a 2048 tirages + jambe feature-z par flip de signe intra-paire + controle d'edition). Le tableau ci-dessous aligne les couches : si le verdict INCONCLUSIVE est un artefact de profondeur, une couche doit s'en ecarter **sur l'ensemble des jambes**, pas sur une seule.

In [4]:
# Mesure par couche -- TRAINED.
res_t = {}
for tag in layers:
    res_t[tag] = measure_humor_differential(
        ROOT / "traces" / f"humor35b_qwen35-2b-base_layer{layers[tag]}of{n_layers_model}_trained.npz",
        pairs, n_draws=n_draws, seed=seed,
    )
    r = res_t[tag]
    print(f"{tag:5s} L{layers[tag]:2d}: delta_pair={r['delta_pair']:.4f} p99={r['null_p99']:.4f} "
          f"ratio_ctrl={r['ratio_vs_ctrl']:.4f} n|z|>3={r['n_features_over3']} verdict={r['verdict']}")

early L 4: delta_pair=18.4938 p99=22.8993 ratio_ctrl=0.9526 n|z|>3=0 verdict=INCONCLUSIVE


mid   L12: delta_pair=27.8327 p99=35.8102 ratio_ctrl=0.9726 n|z|>3=0 verdict=INCONCLUSIVE


late  L19: delta_pair=90.9880 p99=121.4721 ratio_ctrl=0.9564 n|z|>3=1 verdict=INCONCLUSIVE


### Le tableau trained lu en deux colonnes jumelles

Le tableau trained se lit en deux colonnes jumelles : `delta_pair` (l'ecart moyen humour->unfun) et `p99` du null croise (l'ecart que produit le hasard sur les memes donnees). De 18,5 a 27,8 puis 91,0 d'une part ; de 22,9 a 35,8 puis 121,5 d'autre part. **Les deux colonnes font le meme voyage** : le differentiel mesure grossit d'un facteur ~4,9 entre early et late, et le bruit de reference grossit de ~5,3 dans le meme temps. C'est attendu : le residual stream **accumule** les contributions de chaque bloc, sa norme croit mecaniquement avec la profondeur, et tout ecart mesure dessus croit avec elle. Un delta qui grossit seul ne prouverait rien ; c'est precisement pour cela que le protocole pre-registre compare le delta a un null **croise** construit sur les memes vecteurs -- et le rapport des deux reste sous le seuil a chaque profondeur : 18,49/22,90 = 0,81 en early, 27,83/35,81 = 0,78 en mid, 90,99/121,47 = 0,75 en late. Le differentiel humour->unfun ne se separe jamais du bruit que produit le hasard sur les memes vecteurs.

Deux details du tableau meritent l'arret :

- **`n|z|>3 = 1` a la couche late (trained)** : une seule feature depasse trois ecarts-types du null. Le critere pre-registre en exige **au moins trois, du meme cote** ; une isolee dans un dictionnaire de 32 768 atomes est de l'ordre de ce que produit le balayage multiple -- elle ne soutient pas une lecture systematique, et le verdict rendu (INCONCLUSIVE) l'ignore a bon droit.
- **Les verdicts des trois couches sont identiques a celui de la tranche 1** (mid) : la profondeur ajoute de l'amplitude, jamais de la separabilite.

In [5]:
# Mesure par couche -- CONTROL (permutation des input embeddings).
res_c = {}
for tag in layers:
    res_c[tag] = measure_humor_differential(
        ROOT / "traces" / f"humor35b_qwen35-2b-base_layer{layers[tag]}of{n_layers_model}_control.npz",
        pairs, n_draws=n_draws, seed=seed,
    )
    r = res_c[tag]
    print(f"{tag:5s} L{layers[tag]:2d}: delta_pair={r['delta_pair']:.4f} p99={r['null_p99']:.4f} "
          f"ratio_ctrl={r['ratio_vs_ctrl']:.4f} n|z|>3={r['n_features_over3']} verdict={r['verdict']}")

early L 4: delta_pair=18.8027 p99=21.9155 ratio_ctrl=0.9913 n|z|>3=0 verdict=INCONCLUSIVE


mid   L12: delta_pair=22.0069 p99=27.6215 ratio_ctrl=0.9622 n|z|>3=0 verdict=INCONCLUSIVE


late  L19: delta_pair=70.0960 p99=89.7911 ratio_ctrl=0.9690 n|z|>3=0 verdict=INCONCLUSIVE


### Deux controles distincts, deux roles distincts

Le protocole distingue deux controles que la lecture ne doit pas confondre :

- **Le controle d'edition** (`ctrl_edit`, troisieme prompt de chaque paire) fixe l'echelle `delta_ctrl` du verdict : un changement lexical banal, dans la meme zone que la chute. Le critere pre-registre exige que le differentiel humour depasse 1,5 fois cette echelle pour pouvoir etre qualifie. Or le tableau trained affiche des `ratio_vs_ctrl` entre 0,95 et 0,97 : **le differentiel humour->unfun est plus petit que celui d'un banal changement lexical**, a toutes les profondeurs. Le critere 1,5x n'est jamais approche.
- **La permutation des embeddings** (jambe ci-dessus) repond a une autre question : un texte dont l'identite des tokens est detruite -- chaque token recoit l'embedding d'un autre token, la distribution des vecteurs restant la meme -- produit-il le meme profil de mesure ? Le tableau control repond oui dans l'ordre de grandeur : ses delta croissent avec la profondeur (18,8 -> 22,0 -> 70,1) et ses propres ratios restent voisins de 1. Un lecteur qui ne distinguerait pas humour et unfun ne peut pas non plus distinguer humour et texte permute : c'est la signature attendue d'une absence de signal, pas d'un signal faible.

In [6]:
# Identite des top features a travers la profondeur : Jaccard top-5 trained entre couches,
# et trained-vs-control par couche (la disjonction constatee en tranche 1 au mid).
def top_ids(r, n=5):
    return {fid for fid, _ in r["top_features"][:n]}

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a or b) else 0.0

tags = list(layers)
print("Jaccard top-5 trained, entre couches :")
for i in range(len(tags)):
    for j in range(i + 1, len(tags)):
        print(f"  {tags[i]:5s} vs {tags[j]:5s}: {jaccard(top_ids(res_t[tags[i]]), top_ids(res_t[tags[j]])):.2f}")
print("Jaccard top-5 trained vs control, par couche :")
for tag in tags:
    print(f"  {tag:5s}: {jaccard(top_ids(res_t[tag]), top_ids(res_c[tag])):.2f}")
print("Top-5 trained par couche :")
for tag in tags:
    print(f"  {tag:5s}: {[fid for fid, _ in res_t[tag]['top_features'][:5]]}")

Jaccard top-5 trained, entre couches :
  early vs mid  : 0.00
  early vs late : 0.00
  mid   vs late : 0.00
Jaccard top-5 trained vs control, par couche :
  early: 0.00
  mid  : 0.00
  late : 0.00
Top-5 trained par couche :
  early: [27906, 15486, 28565, 8554, 23110]
  mid  : [6677, 12444, 23779, 18944, 2919]
  late : [13173, 32207, 17506, 24480, 14080]


### Les Jaccard sous la loupe : ce que 0,00 veut dire -- et ce qu'il ne veut pas dire

Trois familles de recouvrement, toutes nulles : entre couches (trained), et trained-vs-control a chaque couche. Deux precautions de lecture avant d'en conclure :

- **Un Jaccard top-5 de 0,00 n'est pas, a lui seul, une disjonction remarquable.** Deux top-5 tires au hasard dans un dictionnaire de 32 768 atomes se recouvrent avec une probabilite de l'ordre de 10^-3 : l'esperance du Jaccard sous hasard pur est ~10^-4. La disjonction constatee ici n'a donc de sens que **combinée a l'hypothese qu'elle teste** : si une signature humour transportable existait dans ce dictionnaire, elle devrait apparaitre dans le top-5 de **plusieurs** profondeurs (ou de la meme profondeur entre trained et une permutation qui, par construction, ne la produit pas) -- et donner un Jaccard non nul quelque part. Elle n'apparait nulle part.
- **L'identite des features change integralement d'un etage a l'autre** (les trois listes de la sortie n'ont aucun atome commun). Chaque profondeur du residu est lue par des atomes differents du meme dictionnaire : il n'existe pas de "motif humour" localise qui survivrait au changement de profondeur, meme affaibli.

C'est la **coherence** entre les trois jambes qui porte la conclusion : une amplitude qui ne se separe pas du null (jambe 1), un ratio qui ne depasse jamais le banal lexical (jambe 2), et une identite qui ne se transporte pas (jambe 3). Chaque jambe, seule, serait compatible avec une excuse -- bruit mal estime, echelle mal choisie, rang trop sensible. Les trois ensemble bornent : ce lecteur SAE, a ces profondeurs, ne voit pas de shift systematique humour->unfun.

## Lecture 3 -- Ce que les trois couches disent ensemble

Trois constats, jambe par jambe :

- **Amplitude et null grandissent ensemble.** delta_pair passe de 18.5 (early) a 27.8 (mid) a 91.0 (late) -- l'echelle du residu croit avec la profondeur -- mais le null croise fait exactement de meme (22.9 -> 35.8 -> 121.5) : le rapport delta_pair/p99 reste sous le seuil a toutes les profondeurs (18,49/22,90 = 0,81 ; 27,83/35,81 = 0,78 ; 90,99/121,47 = 0,75). Le differentiel humour->unfun ne se separe jamais du bruit permute, ni tot ni tard.
- **Une seule feature depasse |z|>3 sur toute la experience** (trained late, 1 feature -- seuil de verdict a >=3 non atteint ; le controle de la meme couche en a 0). Isolee, elle ne soutient pas une lecture systematique.
- **Top-5 disjoints partout** : Jaccard 0.00 entre les trois couches trained, et 0.00 trained-vs-control a chaque couche. Aucune signature ne se deplace avec la profondeur ; a chaque etage, le SAE lit des features differentes, et differentes de ce que lit la permutation.

**Verdict etendu : INCONCLUSIVE stable a toutes les profondeurs mesurees.** La tranche 1 ne mesurait qu'un etage ; cette tranche borne desormais early/mid/late : aucun etage du residual stream (4, 12, 19 de 24), lu par ce SAE W32K-L0_50, ne montre de shift systematique humour->unfun au-dessus du bruit. La croissance monotone de l'amplitude est un effet d'echelle du residu, pas un signal -- c'est precisement ce que le null croise controle.

Note de provenance : les traces portent le manifeste canonique du contrat v1 (`instrument='sae'`, `contract_version='v1.1.0'`) -- le champ est lu directement au chargement, sans avertissement de retro-compatibilite.
L'extracteur GPU local ne pose pas encore le champ a la source : les manifestes canoniques sont poses par la migration (`scripts/migrate_trace_manifests.py`).

**Le cadre de falsification, rappele avant le verdict.** Le protocole pre-registre nommait ce qui aurait compte comme un signal : `delta_pair` au-dessus du p99 du null croise, **et** au-dela de 1,5 fois le controle d'edition, **et** au moins trois features a plus de 3 ecarts-types du meme cote. Aucune des trois couches ne satisfait un seul de ces trois criteres -- la couche late en satisfait un tiers de troisieme (une feature isolee). Un INCONCLUSIVE obtenu avec les criteres fixes **avant** la mesure, sur trois profondeurs d'un coup, n'est pas un non-resultat : c'est la borne la plus solide que cet appareil puisse produire.

## Exercices -- sonder les trois faiblesses d'un verdict moyen

Le verdict ci-dessus repose sur des moyennes et des classements. Les trois exercices suivants attaquent chacun une faiblesse distincte de cette lecture : la moyenne qui peut masquer une minorite de paires tres discriminees (exercice 1), la sensibilite du classement top-5 au choix de k (exercice 2), et la tendance du ratio d'echelle avec la profondeur (exercice 3). Les reponses attendues sont courtes : un nombre, une comparaison, une phrase.

In [7]:
# EXERCICE 1 -- delta par paire a la couche late (dispersion cachee par la moyenne)
#
# delta_pair est une MOYENNE sur 30 paires. Une minorite de paires tres discriminees
# peut survivre sous une moyenne tiree par la masse. Recalculez le delta par paire
# (norme L1 de la difference des vecteurs de zone moyennes humour vs unfun) pour
# la couche late, et rapportez la fraction des paires au-dessus de la mediane.
#
# Indice : la zone punchline commence apres le prefixe commun -- _zone_vec(entry, start, d_sae)
# fait deja ce travail pour une entree ; il faut la difference humour vs unfun par paire.
# Etape 1 : pour chaque paire, moyenne des vecteurs de zone des prompts humour puis unfun.
# Etape 2 : norme L1 de la difference, collectee sur les 30 paires.
# Etape 3 : mediane, fraction au-dessus, histogramme 10 bins.
late_tr = np.load(ROOT / "traces" / f"humor35b_qwen35-2b-base_layer{layers['late']}of{n_layers_model}_trained.npz", allow_pickle=False)
deltas_pairs = None  # TODO etudiant : liste de 30 floats
frac_above = None  # TODO etudiant : fraction > mediane
print("Exercice a completer")

Exercice a completer


In [8]:
# EXERCICE 2 -- stabilite du classement : top-10 au lieu de top-5
#
# Le Jaccard top-5 est sensible au moindre echange de rang. Refaites la comparaison
# entre couches avec n=10 : le recouvrement grandit-il (classement stable mais etale),
# ou reste-t-il nul (chaque profondeur lit des features differentes) ?
#
# Indice : top_ids(res, n) accepte n ; il suffit de refaire les trois paires de couches.
# Etape 1 : Jaccard top-10 pour les trois paires (early-mid, early-late, mid-late).
# Etape 2 : comparer aux valeurs top-5 ci-dessus et conclure en une phrase.
jac10 = None  # TODO etudiant : dict {(tag_a, tag_b): jaccard_top10}
print("Exercice a completer")

Exercice a completer


In [9]:
# EXERCICE 3 -- le ratio_vs_ctrl se resserre-t-il avec la profondeur ?
#
# ratio_vs_ctrl compare le delta observe a celui du controle d'edition. En tranche 1
# (mid), il valait ~0.97 trained. Recensez ce ratio par couche (tableau a 3 lignes),
# et repondez : la profondeur rapproche-t-elle l'amplitude de celle d'un banal
# changement lexical, ou l'en eloigne-t-elle ?
#
# Indice : res_t[tag]['ratio_vs_ctrl'] contient deja la valeur par couche.
# Etape 1 : tableau (couche, ratio) pour trained.
# Etape 2 : une phrase d'interpretation -- tendance monotone ou non.
table_ratio = None  # TODO etudiant : liste de tuples (tag, ratio)
print("Exercice a completer")

Exercice a completer


## Conclusion et voir aussi

**Ce que cette tranche etablit** : la stabilite (ou l'instabilite) du verdict INCONCLUSIVE de la tranche 1 a travers trois profondeurs du meme residu, sur le meme corpus et le meme protocole. Un INCONCLUSIVE stable a toutes les couches borne davantage qu'un INCONCLUSIVE ponctuel : aucune profondeur du residual stream, lue par ce SAE, ne montre de shift systematique humour->unfun au-dessus du bruit permute.

**Ce qu'elle n'etablit pas** : l'absence de signature humour dans le modele (d'autres lecteurs que le residu -- MLP, attention, J-lens -- restent hors de portee de cet appareil), et tout effet causal (phase interventionnelle = tranche suivante, cf #14035).

- **ICT-35** (pilote) -- banc humour sur sorties, hors interne.
- **ICT-35b** (tranche 1) -- protocole, pre-enregistrement, verdict mid 12/24.
- `ict/humor_pairs.py` -- corpus, paires, mesure pre-registree (docstrings).
- Issue **#14035** -- protocole complet et ecarts documentes.

**Pourquoi un INCONCLUSIVE *stable* vaut plus qu'un INCONCLUSIVE ponctuel.** L'objection naturelle contre la tranche 1 etait : « mauvaise profondeur ». Cette tranche fait varier ce seul parametre -- meme corpus, meme paires, meme graine, meme SAE, meme protocole -- et l'objection tombe : aucune des trois profondeurs (4, 12, 19 sur 24) ne separe le differentiel du bruit. Ce qui reste ouvert est d'une autre nature : le lecteur (ce SAE W32K-L0_50 en top-k=50), pas la profondeur. Une signature logee dans les MLP, les têtes d'attention ou un autre espace de lecture resterait invisible ici -- c'est le champ des tranches suivantes de #14035, dont la phase interventionnelle (activer/etouffer les features candidates et mesurer l'effet sur le verdict du banc de sortie) trancherait la causalite que cette tranche ne peut qu'encadrer.

## À retenir

1. Le design ne fait varier **qu'un paramètre** : trois profondeurs du même residual stream (couches 4, 12, 19 sur 24 ; fractions 0.1739, 0.5217, 0.8261), à corpus, paires, graine, SAE et protocole identiques. Chaque capture charge 90 prompts et 1868 tokens avec `L0 = 50.00`.
2. `delta_pair` et le null croisé **croissent ensemble** : 18.4938 contre `p99 = 22.8993` en early, 27.8327 contre 35.8102 en mid, 90.9880 contre 121.4721 en late. Les rapports valent 0.81 / 0.78 / 0.75 — sous 1 aux trois profondeurs. Un delta qui grossit seul ne prouverait rien, puisque la norme du residual stream croît mécaniquement avec la profondeur.
3. `ratio_vs_ctrl` vaut 0.9526 / 0.9726 / 0.9564 : le différentiel humour reste partout plus petit que celui d'une édition lexicale banale, loin du seuil 1,5×. `n|z|>3` vaut 0 / 0 / 1 — la seule feature au-delà de 3 écarts-types apparaît en late, sans les deux autres critères pré-enregistrés.
4. Le recouvrement top-5 entre couches est **nul** (Jaccard 0.00) : les features qui bougent le plus ne sont pas les mêmes d'une profondeur à l'autre, ce qui est la signature du bruit et non celle d'un signal stable.
5. Un `INCONCLUSIVE` **stable** aux trois profondeurs (4, 12, 19) écarte l'objection de profondeur ; le paramètre ouvert restant est le **lecteur** — l'argument complet est en conclusion.

## Vérifiez votre compréhension

1. Pourquoi un `delta_pair` qui grossit avec la profondeur ne prouve-t-il rien à lui seul ?
2. Pourquoi comparer à un null **croisé** plutôt qu'à un null par échange d'étiquettes ?
3. Que signifie un Jaccard de 0.00 entre les top-5 de deux couches ?

<details>
<summary>Réponses</summary>

1. Parce que le residual stream **accumule** les contributions de chaque bloc : sa norme croît mécaniquement avec la profondeur, et tout écart mesuré dessus croît avec elle. Ici le null croisé, construit sur les mêmes vecteurs, grossit du même facteur (delta ×4,9 entre early et late, bruit ×5,3) — d'où un rapport qui reste sous le seuil.
2. Parce que le swap d'étiquettes **dégénère** la statistique L1 : les deux groupes auraient alors la même distribution de distances et le test perdrait sa cible. Le null croisé — brasser les unfun entre paires, au moins 2048 tirages — conserve la structure des distances tout en détruisant l'appariement, ce qui est exactement l'hypothèse nulle visée.
3. Que les features les plus déplacées à une couche ne sont pas celles des autres : aucun des 5 rangs n'est commun. Un effet réel laisserait une trace au moins partiellement recouvrante d'une profondeur à l'autre ; un recouvrement nul indique des déplacements distincts d'une couche à l'autre, donc non systématiques.

</details>

## Pour aller plus loin

- **Carnet suivant** : [ICT-35d — le verdict survit-il au découpage par type ?](ICT-35d-HumorTypologyBreakdown-SAE-Python.ipynb), tranche finale : l'annotation typologique figée avant mesure, et 18 jambes au lieu d'une agrégée.
- **Carnet précédent** : [ICT-35b — paires minimales sur substrat SAE](ICT-35b-HumorCausalPairs-SAE-Python.ipynb), dont ce carnet fait varier la seule profondeur.
- **Banc source** : [GameTheory-18c — Humour, banc consolidé](../../GameTheory/GameTheory-18c-Humour-Banc-Python.ipynb).